In [1]:
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
notes_val = pd.read_csv("notes_val_predictions_task1.csv")
ecg_val = pd.read_csv("ecg_val_predictions_task1.csv")
structured_val = pd.read_csv("structured_val_predictions_task1.csv")

display(notes_val.head())
display(ecg_val.head())
display(structured_val.head())

print(len(notes_val))
print(len(ecg_val))
print(len(structured_val))

,sample_id,y_true,pred_prob_notes
0,0,1,0.828592
1,1,1,0.892656
2,2,0,0.504638
3,3,1,0.652006
4,4,1,0.412353


,sample_id,y_true,pred_prob_ecg
0,0,1,0.802370
1,1,1,0.582122
2,2,0,0.725909
3,3,1,0.426241
4,4,1,0.551031


,sample_id,y_true,pred_prob_structured
0,0,1,0.893304
1,1,1,0.713969
2,2,0,0.716149
3,3,1,0.918504
4,4,1,0.541204


16262
16262
16262


In [3]:
val_df = notes_val.merge(
    ecg_val[["sample_id", "pred_prob_ecg"]],
    on="sample_id",
    how="inner"
).merge(
    structured_val[["sample_id", "pred_prob_structured"]],
    on="sample_id",
    how="inner"
)

val_df

,sample_id,y_true,pred_prob_notes,pred_prob_ecg,pred_prob_structured
0,0,1,0.828592,0.802370,0.893304
1,1,1,0.892656,0.582122,0.713969
2,2,0,0.504638,0.725909,0.716149
3,3,1,0.652006,0.426241,0.918504
4,4,1,0.412353,0.551031,0.541204
...,...,...,...,...,...
16257,16257,1,0.908658,0.445923,0.872565
16258,16258,1,0.661568,0.713634,0.525163
16259,16259,1,0.807309,0.663995,0.865208
16260,16260,1,0.220007,0.283680,0.361295


In [4]:
notes_test = pd.read_csv("notes_test_predictions_task1.csv")
ecg_test = pd.read_csv("ecg_test_predictions_task1.csv")
structured_test = pd.read_csv("structured_test_predictions_task1.csv")

display(notes_test.head())
display(ecg_test.head())
display(structured_test.head())

print(len(notes_test))
print(len(ecg_test))
print(len(structured_test))

,sample_id,y_true,pred_prob_notes
0,0,1,0.907967
1,1,1,0.625266
2,2,0,0.738681
3,3,1,0.490144
4,4,1,0.839175


,sample_id,y_true,pred_prob_ecg
0,0,1,0.630932
1,1,1,0.679002
2,2,0,0.366071
3,3,1,0.584326
4,4,1,0.779573


,sample_id,y_true,pred_prob_structured
0,0,1,0.428050
1,1,1,0.495168
2,2,0,0.079507
3,3,1,0.552566
4,4,1,0.720961


16262
16262
16262


In [5]:
test_df = notes_test.merge(
    ecg_test[["sample_id", "pred_prob_ecg"]],
    on="sample_id",
    how="inner"
).merge(
    structured_test[["sample_id", "pred_prob_structured"]],
    on="sample_id",
    how="inner"
)
test_df

,sample_id,y_true,pred_prob_notes,pred_prob_ecg,pred_prob_structured
0,0,1,0.907967,0.630932,0.428050
1,1,1,0.625266,0.679002,0.495168
2,2,0,0.738681,0.366071,0.079507
3,3,1,0.490144,0.584326,0.552566
4,4,1,0.839175,0.779573,0.720961
...,...,...,...,...,...
16257,16257,0,0.772949,0.285153,0.540499
16258,16258,1,0.858452,0.803755,0.817050
16259,16259,0,0.360250,0.433835,0.365633
16260,16260,1,0.709560,0.802207,0.811998


In [6]:
# Feature engineering
def add_meta_features(df):
    eps = 1e-6
    
    # base probs
    s = df["pred_prob_structured"].astype(float)
    n = df["pred_prob_notes"].astype(float)
    e = df["pred_prob_ecg"].astype(float)
    
    # interaction terms
    df["notes_x_ecg"] = n * e
    df["notes_x_struct"] = n * s
    df["ecg_x_struct"] = e * s
    df["notes_x_ecg_x_struct"] = n * e * s
    
    # pairwise differences
    df["struct_minus_notes"] = s - n
    df["struct_minus_ecg"] = s - e
    df["notes_minus_ecg"] = n - e
    
    # summary stats across modalities
    df["prob_mean"] = (s + n + e) / 3.0
    df["prob_max"] = np.maximum(np.maximum(s, n), e)
    df["prob_min"] = np.minimum(np.minimum(s, n), e)
    df["prob_range"] = df["prob_max"] - df["prob_min"]
    
    # agreement / dispersion
    df["prob_std"] = np.std(np.vstack([s, n, e]), axis=0)
    
    # logit transform can help logistic stacking
    df["logit_struct"] = np.log((s + eps) / (1 - s + eps))
    df["logit_notes"] = np.log((n + eps) / (1 - n + eps))
    df["logit_ecg"] = np.log((e + eps) / (1 - e + eps))
    
    # rank-like indicators / thresholds
    df["struct_gt_notes"] = (s > n).astype(int)
    df["struct_gt_ecg"] = (s > e).astype(int)
    df["notes_gt_ecg"] = (n > e).astype(int)
    
    # confidence-style features
    df["struct_conf"] = np.abs(s - 0.5)
    df["notes_conf"] = np.abs(n - 0.5)
    df["ecg_conf"] = np.abs(e - 0.5)
    
    return df

val_df = add_meta_features(val_df.copy())
test_df = add_meta_features(test_df.copy())

# Features
stack_features = [
    # raw probs
    "pred_prob_structured",
    "pred_prob_notes",
    "pred_prob_ecg",
    
    # interactions
    "notes_x_ecg",
    "notes_x_struct",
    "ecg_x_struct",
    "notes_x_ecg_x_struct",
    
    # differences
    "struct_minus_notes",
    "struct_minus_ecg",
    "notes_minus_ecg",
    
    # summaries
    "prob_mean",
    "prob_max",
    "prob_min",
    "prob_range",
    "prob_std",
    
    # logits
    "logit_struct",
    "logit_notes",
    "logit_ecg",
    
    # comparisons
    "struct_gt_notes",
    "struct_gt_ecg",
    "notes_gt_ecg",
    
    # confidence
    "struct_conf",
    "notes_conf",
    "ecg_conf",
]

X_val = val_df[stack_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_val = val_df["y_true"].astype(int)

X_test = test_df[stack_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_test = test_df["y_true"].astype(int)

# Logistic stacking model
meta_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.2,
        C=0.5,
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    ))
])

meta_model.fit(X_val, y_val)

# Predict + evaluate
test_df["pred_logistic_ensemble"] = meta_model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, test_df["pred_logistic_ensemble"])
auprc = average_precision_score(y_test, test_df["pred_logistic_ensemble"])

print("Logistic multimodal test AUROC:", round(auc, 6))
print("Logistic multimodal test AUPRC:", round(auprc, 6))

# Inspect learned weights
lr = meta_model.named_steps["lr"]
coefs = pd.DataFrame({
    "feature": stack_features,
    "coef": lr.coef_[0]
}).sort_values("coef", ascending=False)

print("\nTop positive coefficients:")
print(coefs.head(10).to_string(index=False))

print("\nTop negative coefficients:")
print(coefs.tail(10).to_string(index=False))

Logistic multimodal test AUROC: 0.798308
Logistic multimodal test AUPRC: 0.843603

Top positive coefficients:
        feature     coef
   logit_struct 1.849795
      logit_ecg 0.678682
notes_minus_ecg 0.315421
     notes_conf 0.120571
pred_prob_notes 0.035724
struct_gt_notes 0.024201
    struct_conf 0.020711
       prob_max 0.009511
     prob_range 0.004535
       ecg_conf 0.003012

Top negative coefficients:
             feature      coef
       struct_gt_ecg -0.019155
        notes_gt_ecg -0.020412
pred_prob_structured -0.025313
        ecg_x_struct -0.052282
            prob_std -0.064665
       pred_prob_ecg -0.091848
         logit_notes -0.154436
         notes_x_ecg -0.221319
  struct_minus_notes -0.350596
notes_x_ecg_x_struct -0.508530
